# 1. [calculate] labour productivity
- Save to `working_yearly` new table
**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
fame_yearly = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly = fame_yearly.mutate(
    gva1 = fame_yearly.wages + fame_yearly.ebitda,
    gva2 =  fame_yearly.profit_loss_pretax +
            fame_yearly.interest_paid +
            fame_yearly.depreciation +
            fame_yearly.remuneration_employees
).mutate(
    gva1_per_worker = ibis._.gva1 / fame_yearly.employees,
    gva2_per_worker = ibis._.gva2 / fame_yearly.employees,
    average_wage = fame_yearly.wages / fame_yearly.employees
)

# Verify the final materialized table
table_t_working = ibis.memtable(working_yearly)
print(f"\nSample of {new_table_name}:")
display(table_t_working.sample(0.0001).execute())


Sample of working_yearly:


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda,gva1,gva2,gva1_per_worker,gva2_per_worker,average_wage
0,02236262,2006,False,4554.619681,368.375164,12.034426,11,1494.780448,1486.593002,NaN,...,54.604920,18.846004,NaN,158.363,46.518505,484.046857,487.997702,44.004260,44.363427,39.775305
1,00638328,2006,False,6419.768155,794.825540,744.934621,70,263.303727,NaN,NaN,...,150.916603,14.419813,NaN,NaN,1171.960467,2705.582385,NaN,38.651177,NaN,21.908885
2,02838588,2006,False,13047.941736,9186.587316,3176.834733,91,2613.137029,1527.181797,NaN,...,450.047727,NaN,NaN,32.000,3708.332897,8050.758763,NaN,88.469877,NaN,47.718966
3,03456326,2006,True,118271.430657,-35626.985214,-13470.557745,176,2287.029772,NaN,NaN,...,3626.558547,572.183480,NaN,397.325,-19106.380692,-3053.438672,6140.752302,-17.349083,34.890638,91.209898
4,00918093,2006,False,NaN,829.136190,-212.837337,17,184.264049,180.814379,NaN,...,30.294343,8.037674,NaN,NaN,-1634.801070,-1293.958285,NaN,-76.115193,NaN,20.049576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,12644682,2023,False,11861.887921,1299.482495,495.402473,65,1946.420001,NaN,NaN,...,201.016934,73.549712,NaN,101.263,770.167062,2883.712139,3025.011196,44.364802,46.538634,32.516078
112,13551769,2023,True,35181.879271,21523.560350,588.616358,179,3372.928558,2337.791560,2291.454238,...,651.650244,289.878049,NaN,2005.243,780.660962,7173.269807,8201.468496,40.074133,45.818260,35.712899
113,07706662,2023,False,1611.447440,642.416817,-14.350668,27,17.091550,NaN,NaN,...,NaN,NaN,NaN,NaN,-10.084534,813.987442,NaN,30.147683,NaN,30.521184
114,12178578,2024,False,10085.236000,767.472000,-1158.764000,368,3159.613000,NaN,NaN,...,453.980000,802.578000,NaN,114.291,-258.732000,6174.887000,7502.655000,16.779584,20.387649,17.482660


In [ ]:
working_yearly_skinny = table_t_working.select(
    "registered_number", "year",
    "employees", "fixed_total", "total_assets",
    "average_wage", "gva1", "gva2",
    "gva1_per_worker", "gva2_per_worker"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_skinny.columns}")
con.create_table(new_table_name, working_yearly_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nHead of {new_table_name}:")
display(final_table.sample(200 / row_count).execute())

✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'employees', 'fixed_total', 'total_assets', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,128,490
📊 Number of columns: 10

Head of working_yearly:


,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker
0,SC237955,2006,80,6289.908921,10129.661917,46.829168,4452.727317,7359.325371,55.659091,91.991567
1,02904192,2006,65,2031.971827,4155.274750,33.107698,2275.866851,2214.119069,35.013336,34.063370
2,00362221,2006,466,14683.166920,91607.006070,28.557853,26526.435939,27956.968385,56.923682,59.993494
3,02575859,2006,19,232.644555,993.490539,27.462162,-286.950354,NaN,-15.102650,NaN
4,03551717,2006,32,5287.619203,12497.721076,49.620354,3241.802155,3614.784769,101.306317,112.962024
...,...,...,...,...,...,...,...,...,...,...
193,03178478,2023,120,1018.220000,16462.955000,53.308126,8600.694117,9407.421421,71.672451,78.395179
194,03045939,2023,37,6.268287,583.245767,29.842365,1232.833114,NaN,33.319814,NaN
195,02969165,2024,14,NaN,294.470000,30.780286,441.736000,NaN,31.552571,NaN
196,00365812,2024,617,42841.000000,47618.000000,9.134522,6240.000000,NaN,10.113452,NaN


# [calculate] 2. TFP

$$\ln(Y_{it}) = \alpha_i + \gamma_t + \beta_K \ln(K_{it}) + \beta_L \ln(L_{it}) + \varepsilon_{it}$$
- $Y_{it}$: `gva1` or `gva2`
- $K_{it}$: `fixed_total` or `total_assets`
- $L_{it}$: `employees`  
### Capital choice
- `fixed_total` = `tangibles` + `intangibles` + `investments_other`, representing different types of capitals
- `total_assets` = `fixed_total` + `current_assets`, which includes non-productive current_assets (e.g. cash, stock, debtors) and productive current_assets (e.g. stock of raw materials, work in progress and finished goods).

In [15]:
import ibis
from ibis import _
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(str(dirs.db_path))
table_working = con.table("working_yearly")
table_results = table_working.select("registered_number", "year")

# 3. Define the 4 model setups to iterate through
# Varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'tfp1_ft': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees', 'firm_fe': True, 'time_fe': True, 'description': 'GVA = renumeration + EBITDA, K = total assets, firm and time fixed effects'},
    'tfp2_ft': {'Y': 'gva2', 'K': 'total_assets', 'L': 'employees', 'firm_fe': True, 'time_fe': True, 'description': 'GVA = renumeration + pnl + interest + depreciation, K = total assets, firm and time fixed effects'},
    'tfp3_ft': {'Y': 'gva1', 'K': 'fixed_total', 'L': 'employees', 'firm_fe': True, 'time_fe': True, 'description': 'GVA = renumeration + EBITDA, K = fixed assets (no current assets), firm and time fixed effects'},
    'tfp1_f': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees', 'firm_fe': True, 'time_fe': False, 'description': 'GVA = renumeration + EBITDA, K = total assets, firm only fixed effects'},
    'tfp1_t': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees', 'firm_fe': False, 'time_fe': True, 'description': 'GVA = renumeration + EBITDA, K = total assets, time only fixed effects'},
    'tfp1_n': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees', 'firm_fe': False, 'time_fe': False, 'description': 'GVA = renumeration + EBITDA, K = total assets, no fixed effects'},
}

# Dictionary to store the parameter outputs (\beta_K, \beta_L) and model summaries
parameter_tables = {}

for name, mod in models.items():

    # Mutate to dynamically log-transform all columns, adding ln_ prefix to column name
    model_params = ['Y', 'K', 'L']
    rename_dict = { key: value for key, value in mod.items() if key in model_params }
    table_skinny = (
        table_working
        .select(["registered_number", "year"] + list(rename_dict.values()))
        .rename(rename_dict)
    )
    table_start = table_skinny
    for col in model_params:
        table_filtered = table_start.filter(_[col] > 0)
        table_logged = table_filtered.mutate(**{f'ln_{col}': np.log(_[col]) }) # type: ignore
        table_start = table_logged

    # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
    df_model = table_start.execute().set_index(['registered_number', 'year'])

    # Define Endogenous (Y) and Exogenous (X) variables
    Y = df_model['ln_Y']
    X = sm.add_constant(df_model[['ln_K', 'ln_L']])
    
    # 2. Estimate the model with Firm and Year Fixed Effects
    mod_ols = PanelOLS(Y, X, entity_effects=mod['firm_fe'], time_effects=mod['time_fe'])
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta_K and \beta_L
    beta_K = res.params['ln_K']
    beta_L = res.params['ln_L']
    
    # 3. Calculate firm-year specific TFP (Solow Residual) inside the Ibis pipeline
    # TFP_it = ln(Y_it) - \beta_K*ln(K_it) - \beta_L*ln(L_it)
    table_with_tfp = table_start.mutate(
        **{name: _['ln_Y'] - (beta_K * _['ln_K']) - (beta_L * _['ln_L'])}
    )
    
    # 4. Join the calculated TFP column back to the main dataframe
    # We select only the keys and the new TFP column to avoid duplicating ln_ columns
    table_results = (
        table_results
        .left_join(
            table_with_tfp,
            ["registered_number", "year"],
            rname='{name}_' + name
        )
        .drop("registered_number_" + name, "year_" + name) # Drop duplicate join keys
    )
    
    # 5. Store the results and parameters
    parameter_tables[name] = {
        'beta_K': beta_K,
        'beta_L': beta_L,
        # Safely extract time effects if they exist
        'gamma_t': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None, 
        'summary': res.summary
    }
    print(f"✅ Model '{name}' estimated: beta_K={beta_K:.4f}, beta_L={beta_L:.4f}")

print(f"Panel regressions complete. {len(models)} TFP variants added to results dataframe.")

✅ Model 'tfp1_ft' estimated: beta_K=0.2864, beta_L=0.6169
✅ Model 'tfp2_ft' estimated: beta_K=0.2654, beta_L=0.6109
✅ Model 'tfp3_ft' estimated: beta_K=0.0651, beta_L=0.7197
✅ Model 'tfp1_f' estimated: beta_K=0.2836, beta_L=0.6135
✅ Model 'tfp1_t' estimated: beta_K=0.4213, beta_L=0.5615
✅ Model 'tfp1_n' estimated: beta_K=0.4212, beta_L=0.5612
Panel regressions complete. 6 TFP variants added to results dataframe.


In [ ]:
# Display the parameter_tables dictionary: name, beta_K, beta_L, description from my models dictionary
# Display it in an easy-to-read table


results_df = pd.DataFrame([
    {
        'name': name,
        'beta_K': params['beta_K'],
        'beta_L': params['beta_L'],
        'description': models[name]['description']
    }
    for name, params in parameter_tables.items()
])
results_df.style.format({
    'beta_K': '{:.3f}',
    'beta_L': '{:.3f}'
})
results_df.style.hide(axis='index')
print(results_df.to_string(index=False))

,name,beta_K,beta_L,description
0,tfp1_ft,0.286384,0.616892,"GVA = renumeration + EBITDA, K = total assets,..."
1,tfp2_ft,0.265447,0.610908,GVA = renumeration + pnl + interest + deprecia...
2,tfp3_ft,0.065087,0.719712,"GVA = renumeration + EBITDA, K = fixed assets ..."
3,tfp1_f,0.283609,0.613465,"GVA = renumeration + EBITDA, K = total assets,..."
4,tfp1_t,0.421262,0.561531,"GVA = renumeration + EBITDA, K = total assets,..."
5,tfp1_n,0.421195,0.561155,"GVA = renumeration + EBITDA, K = total assets,..."


In [ ]:
# From the above cell, display the revised table with the new TFP columns and the parameter estimates for each model
table_sample = table_results.sample(0.0001).execute()
display(table_sample)

# Send parameter_tables to a markdown file in dirs.output_dir to easily compare
output_file = dirs.output_dir / f"tfp_2factor_results.md"
with open(output_file, 'w') as f:
    for name, params in parameter_tables.items():
        # Write all to 1 big markdown file
        # Just write the default display(params) output to the file
        f.write(f"# Parameters for model '{name}'\n\n")
        f.write(f"## Estimated Coefficients\n")
        f.write(f"- beta_K: {params['beta_K']:.6f}\n")
        f.write(f"- beta_L: {params['beta_L']:.6f}\n")
        if params['gamma_t'] is not None:
            f.write(f"\n## Time Effects (gamma_t)\n")
            f.write(params['gamma_t'].to_markdown())
        f.write("\n\n## Model Summary\n")
        f.write(params['summary'].as_text())
    print(f"✅ Parameters for model written to {output_file}")

# Verify that that \ln Y = \alpha_i + \gamma_t + \beta_K \ln K + \beta_L \ln L + TFP_it holds for this sample
name, mod = list(models.items())[0]
params = parameter_tables[name]
for row in table_sample.itertuples():
    # Get TFP, L, K
    tfp = row._asdict()[name]
    ln_K = row.ln_K
    ln_L = row.ln_L
    ln_Y = row.ln_Y
    if any(np.isnan([tfp, ln_K, ln_L, ln_Y])):
        print(f"Skipping row {row.Index} due to NaN values.")
        continue
    ln_Y_calc = tfp + params['beta_K'] * ln_K + params['beta_L'] * ln_L
    diff = ln_Y - ln_Y_calc

    assert np.isclose(diff, 0, atol=1e-6), (
        f"TFP calculation check failed for row {row.Index}!  \
        Expected ln_Y: {ln_Y:.6f}, Calculated ln_Y: {ln_Y_calc:.6f}, Difference: {diff:.6e}"
    )

# Count number of tfp1 and tfp2 observations in the results table (as a percentage of total rows)
# Output as a pd dataframe
total_rows = table_results.count().execute()
tfp1_count = table_results.filter(_['tfp1'].isnull() == False).count().execute()
tfp2_count = table_results.filter(_['tfp2'].isnull() == False).count().execute()
tfp3_count = table_results.filter(_['tfp3'].isnull() == False).count().execute()
tfp_counts = pd.DataFrame({
    'TFP Variant': ['tfp1', 'tfp2', 'tfp3'],
    'Count': [tfp1_count, tfp2_count, tfp3_count]
})
tfp_counts['Percentage'] = tfp_counts['Count'] / total_rows * 100
print(f"\nTFP Counts and Percentages (out of {total_rows:,} total rows):")
display(tfp_counts)

,registered_number,year,Y,K,L,ln_Y,ln_K,ln_L,tfp1_ft,Y_tfp2_ft,...,ln_K_tfp1_t,ln_L_tfp1_t,tfp1_t,Y_tfp1_n,K_tfp1_n,L_tfp1_n,ln_Y_tfp1_n,ln_K_tfp1_n,ln_L_tfp1_n,tfp1_n
0,04503206,2024,172874.000000,288881.000000,1421.0,12.060318,12.573770,7.259116,3.981303,193889.000000,...,12.573770,7.259116,2.687248,172874.000000,288881.000000,1421.0,12.060318,12.573770,7.259116,2.690824
1,SC154655,2008,35585.390607,273793.692086,508.0,10.479690,12.520130,6.230481,3.050593,35553.581006,...,12.520130,6.230481,1.706827,35585.390607,273793.692086,508.0,10.479690,12.520130,6.230481,1.710013
2,00728599,2008,49075.703380,294705.248921,1107.0,10.801119,12.593731,7.009409,2.870430,48423.242825,...,12.593731,7.009409,1.559859,49075.703380,294705.248921,1107.0,10.801119,12.593731,7.009409,1.563343
3,02675504,2009,165262.523768,517301.833568,41.0,12.015291,13.156382,3.713572,5.956642,157976.892898,...,13.156382,3.713572,4.387720,165262.523768,517301.833568,41.0,12.015291,13.156382,3.713572,4.390004
4,01224533,2009,1471.816427,14585.566996,65.0,7.294253,9.587788,4.174387,1.973319,1647.911065,...,9.587788,4.174387,0.911234,1471.816427,14585.566996,65.0,7.294253,9.587788,4.174387,0.913450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,01695398,2016,3980.118856,14634.540073,52.0,8.289067,9.591140,3.951244,3.104828,NaN,...,9.591140,3.951244,2.029938,3980.118856,14634.540073,52.0,8.289067,9.591140,3.951244,2.032070
82,07059779,2020,11636.662231,10617.131982,219.0,9.361916,9.270224,5.389072,3.382598,NaN,...,9.270224,5.389072,2.430592,11636.662231,10617.131982,219.0,9.361916,9.270224,5.389072,2.433243
83,07831507,2018,3457.108591,29095.499989,106.0,8.148188,10.278339,4.663439,2.327799,NaN,...,10.278339,4.663439,1.199649,3457.108591,29095.499989,106.0,8.148188,10.278339,4.663439,1.202095
84,03904201,2008,31402.577309,54916.008633,529.0,10.354645,10.913560,6.270988,3.360655,NaN,...,10.913560,6.270988,2.235823,31402.577309,54916.008633,529.0,10.354645,10.913560,6.270988,2.238916


✅ Parameters for model 'tfp1_n' written to C:\Users\lazyst\Files\ucl\Dissertation\descriptives\output\tfp_2factor_results.md
Skipping row 44 due to NaN values.
Skipping row 51 due to NaN values.
Skipping row 54 due to NaN values.
Skipping row 62 due to NaN values.
Skipping row 69 due to NaN values.
Skipping row 75 due to NaN values.
Skipping row 85 due to NaN values.


IbisTypeError: Column 'tfp1' is not found in table. Existing columns: 'registered_number', 'year', 'Y', 'K', 'L', 'ln_Y', 'ln_K', 'ln_L', 'tfp1_ft', 'Y_tfp2_ft', 'K_tfp2_ft', 'L_tfp2_ft', 'ln_Y_tfp2_ft', 'ln_K_tfp2_ft', 'ln_L_tfp2_ft', 'tfp2_ft', 'Y_tfp3_ft', 'K_tfp3_ft', 'L_tfp3_ft', 'ln_Y_tfp3_ft', 'ln_K_tfp3_ft', 'ln_L_tfp3_ft', 'tfp3_ft', 'Y_tfp1_f', 'K_tfp1_f', 'L_tfp1_f', 'ln_Y_tfp1_f', 'ln_K_tfp1_f', 'ln_L_tfp1_f', 'tfp1_f', 'Y_tfp1_t', 'K_tfp1_t', 'L_tfp1_t', 'ln_Y_tfp1_t', 'ln_K_tfp1_t', 'ln_L_tfp1_t', 'tfp1_t', 'Y_tfp1_n', 'K_tfp1_n', 'L_tfp1_n', 'ln_Y_tfp1_n', 'ln_K_tfp1_n', 'ln_L_tfp1_n', 'tfp1_n'.

In [5]:
preferred_model = 'tfp1'
# Add the tfp1 column from this table_results to the working_yearly table in the database
# Rename as 'tfp'
table_with_tfp = (
    table_working
    .left_join(
        table_results.select(['registered_number', 'year', preferred_model]),
        ['registered_number', 'year']
    )
    # Rename the preferred TFP column to 'tfp' for clarity
    .rename(tfp=preferred_model)
    .drop("registered_number_right", "year_right")
)
display(table_with_tfp.sample(0.0001).execute())

,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker,tfp
0,SC180242,2006,67,6301.795144,13507.000000,48.639187,5965.901425,NaN,89.043305,NaN,3.376188
1,05315339,2006,158,3569.441143,9635.332835,60.957309,13131.581195,12813.904707,83.111273,81.100663,3.732647
2,03705002,2006,80,634.157355,13331.148856,33.909970,3300.931488,2911.916848,41.261644,36.398961,2.678689
3,04847541,2006,10,47.401419,354.062505,87.487711,918.336949,NaN,91.833695,NaN,3.721195
4,01531357,2006,43,3427.372476,6002.471700,28.379249,1495.721550,1411.574779,34.784222,32.827320,2.498591
...,...,...,...,...,...,...,...,...,...,...,...
111,04000753,2024,138,14.751000,16292.634000,44.555768,7230.011000,7879.885000,52.391384,57.100616,3.068928
112,03335009,2024,26,44.548000,7732.059000,163.952462,4021.222000,4678.702000,154.662385,179.950077,3.725415
113,04587255,2024,22,356.201000,15703.217000,76.162955,3772.860000,NaN,171.493636,NaN,3.561817
114,SC279329,2024,14,NaN,NaN,10.053500,155.823000,NaN,11.130214,NaN,NaN


In [6]:
overwrite = True
if overwrite:
    # Safe overwrite of the working_yearly table with the new tfp column
    con.create_table("working_yearly_temp", table_with_tfp)
    con.create_table("working_yearly", con.table("working_yearly_temp"), overwrite=True)
    con.drop_table("working_yearly_temp")

    # Verify schema and rows of the final working_yearly table
    final_table = con.table("working_yearly")
    row_count = final_table.count().execute()
    col_count = len(final_table.columns)
    tfp_col_exists = 'tfp' in final_table.columns
    tfp_nan_count = final_table.filter(_['tfp'].isnull() == True).count().execute() if tfp_col_exists else None
    print(f"\nFinal 'working_yearly' table verification:"
        f"\n- Rows: {row_count:,}"
        f"\n- Columns: {col_count:,}"
        f"\n- TFP column exists: {tfp_col_exists}"
        f"\n- TFP NaN count: {tfp_nan_count:,}")


Final 'working_yearly' table verification:
- Rows: 1,128,490
- Columns: 11
- TFP column exists: True
- TFP NaN count: 83,326


In [7]:
con.raw_sql("CHECKPOINT;")
con.disconnect()